## 1. Сбор данных

Сбор исходных данных из Eurostat API и Weekly Oil Bulletin. Исходные файлы сохраняются в `data/raw/`.

In [1]:
import json
import os
import time

import pandas as pd
import requests

RAW = "data/raw"
os.makedirs(RAW, exist_ok=True)

API = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"

YEARS = list(range(2015, 2025))

EU27 = ["AT", "BE", "BG", "CY", "CZ", "DE", "DK", "EE", "EL", "ES", "FI", "FR",
        "HR", "HU", "IE", "IT", "LT", "LU", "LV", "MT", "NL", "PL", "PT", "RO",
        "SE", "SI", "SK"]


def jsonstat_to_long(js):
    """Ответ Eurostat (JSON-stat 2.0) -> длинная таблица."""
    dims = js["id"]
    sizes = js["size"]

    codes = []
    for d in dims:
        index = js["dimension"][d]["category"]["index"]
        if isinstance(index, dict):
            codes.append(sorted(index, key=index.get))
        else:
            codes.append(list(index))

    values = js["value"]
    if isinstance(values, list):
        pairs = [(i, v) for i, v in enumerate(values) if v is not None]
    else:
        pairs = [(int(k), v) for k, v in values.items()]

    rows = []
    for flat, val in pairs:
        pos, rest = [], flat
        for s in reversed(sizes):
            pos.append(rest % s)
            rest //= s
        pos.reverse()

        row = {d: codes[i][pos[i]] for i, d in enumerate(dims)}
        row["value"] = val
        rows.append(row)

    return pd.DataFrame(rows)


def build_params(filters):
    params = [("format", "JSON"), ("lang", "EN")]
    for key, val in filters.items():
        if isinstance(val, (list, tuple)):
            params += [(key, str(v)) for v in val]
        else:
            params.append((key, str(val)))
    return params



def get_eurostat(dataset, filters, name):
    r = requests.get(f"{API}/{dataset}", params=build_params(filters), timeout=180)
    r.raise_for_status()
    js = r.json()

    with open(f"{RAW}/{name}.json", "w", encoding="utf-8") as f:
        json.dump(js, f, ensure_ascii=False)

    df = jsonstat_to_long(js)
    df.to_csv(f"{RAW}/{name}.csv", index=False)
    time.sleep(1)
    return df


get_eurostat("road_eqr_carpda",
             {"geo": EU27, "time": YEARS},
             "road_eqr_carpda")

get_eurostat("nama_10_pc",
             {"unit": "CP_PPS_EU27_2020_HAB", "na_item": ["B1GQ", "P41"],
              "geo": EU27, "time": YEARS},
             "nama_10_pc")

get_eurostat("ilc_di03",
             {"statinfo": "MED_EI", "unit": "PPS", "age": "TOTAL", "sex": "T",
              "geo": EU27, "time": YEARS},
             "ilc_di03")

get_eurostat("demo_pjan",
             {"unit": "NR", "age": "TOTAL", "sex": "T",
              "geo": EU27, "time": YEARS},
             "demo_pjan")

get_eurostat("nrg_pc_204",
             {"siec": "E7000", "nrg_cons": "KWH2500-4999", "unit": "KWH",
              "tax": "I_TAX", "currency": "EUR", "geo": EU27},
             "nrg_pc_204")

get_eurostat("gov_10a_exp",
             {"unit": "PC_GDP", "sector": "S13", "cofog99": "GF05",
              "na_item": "TE", "geo": EU27, "time": YEARS},
             "gov_10a_exp")

OIL_URL = ("https://energy.ec.europa.eu/document/download/"
           "906e60ca-8b6a-44e7-8589-652854d2fd3f_en"
           "?filename=Weekly_Oil_Bulletin_Prices_History_maticni_4web.xlsx")

r = requests.get(OIL_URL, timeout=300)
r.raise_for_status()
with open(f"{RAW}/weekly_oil_bulletin_history.xlsx", "wb") as f:
    f.write(r.content)

## 2. Объединение и очистка данных

Формирование панели «страна–год», очистка, расчёт производных показателей и сохранение итогового датасета.

In [2]:
import os
import re

import numpy as np
import pandas as pd

RAW = "data/raw"
CLEAN = "data/cleaned"
os.makedirs(CLEAN, exist_ok=True)

YEARS = list(range(2015, 2025))

COUNTRIES = {
    "AT": ("Austria", "Западная Европа"),
    "BE": ("Belgium", "Западная Европа"),
    "BG": ("Bulgaria", "Восточная Европа"),
    "CY": ("Cyprus", "Южная Европа"),
    "CZ": ("Czechia", "Восточная Европа"),
    "DE": ("Germany", "Западная Европа"),
    "DK": ("Denmark", "Северная Европа"),
    "EE": ("Estonia", "Северная Европа"),
    "EL": ("Greece", "Южная Европа"),
    "ES": ("Spain", "Южная Европа"),
    "FI": ("Finland", "Северная Европа"),
    "FR": ("France", "Западная Европа"),
    "HR": ("Croatia", "Южная Европа"),
    "HU": ("Hungary", "Восточная Европа"),
    "IE": ("Ireland", "Северная Европа"),
    "IT": ("Italy", "Южная Европа"),
    "LT": ("Lithuania", "Северная Европа"),
    "LU": ("Luxembourg", "Западная Европа"),
    "LV": ("Latvia", "Северная Европа"),
    "MT": ("Malta", "Южная Европа"),
    "NL": ("Netherlands", "Западная Европа"),
    "PL": ("Poland", "Восточная Европа"),
    "PT": ("Portugal", "Южная Европа"),
    "RO": ("Romania", "Восточная Европа"),
    "SE": ("Sweden", "Северная Европа"),
    "SI": ("Slovenia", "Южная Европа"),
    "SK": ("Slovakia", "Восточная Европа"),
}
NAMES = {k: v[0] for k, v in COUNTRIES.items()}
REGIONS = {k: v[1] for k, v in COUNTRIES.items()}

def read_raw(name):
    return pd.read_csv(f"{RAW}/{name}.csv")

cars = read_raw("road_eqr_carpda")
cars["time"] = cars["time"].astype(int)

available = set(cars["mot_nrg"])
bev_code = next(c for c in ["ELC", "ELC_BAT"] if c in available)
phev_codes = [c for c in ["ELC_PET_PI", "ELC_DIE_PI"] if c in available]

wide = cars.pivot_table(index=["geo", "time"], columns="mot_nrg",
                        values="value", aggfunc="sum")

ev = pd.DataFrame(index=wide.index)
ev["new_cars_total"] = wide["TOTAL"]
ev["new_cars_bev"] = wide[bev_code]
ev["new_cars_phev"] = (wide[phev_codes].sum(axis=1, min_count=1)
                       if phev_codes else np.nan)
ev = ev.reset_index().rename(columns={"geo": "country_code", "time": "year"})

nama = read_raw("nama_10_pc")
nama = nama[(nama["unit"] == "CP_PPS_EU27_2020_HAB") & nama["na_item"].isin(["B1GQ", "P41"])]
nama["time"] = nama["time"].astype(int)
nama = (nama.pivot_table(index=["geo", "time"], columns="na_item",
                         values="value")
            .reset_index()
            .rename(columns={"geo": "country_code", "time": "year",
                             "B1GQ": "gdp_per_capita_pps",
                             "P41": "aic_per_capita_pps"}))

inc = read_raw("ilc_di03")
inc = inc[(inc["statinfo"] == "MED_EI") & (inc["unit"] == "PPS") &
          (inc["age"] == "TOTAL") & (inc["sex"] == "T")]
inc["time"] = inc["time"].astype(int)
inc = inc[["geo", "time", "value"]].rename(
    columns={"geo": "country_code", "time": "year",
             "value": "median_disposable_income"})

pop = read_raw("demo_pjan")
pop = pop[(pop["age"] == "TOTAL") & (pop["sex"] == "T") & (pop["unit"] == "NR")]
pop["time"] = pop["time"].astype(int)
pop = pop[["geo", "time", "value"]].rename(
    columns={"geo": "country_code", "time": "year", "value": "population"})

elec = read_raw("nrg_pc_204")
elec = elec[(elec["nrg_cons"] == "KWH2500-4999") & (elec["unit"] == "KWH") &
            (elec["tax"] == "I_TAX") & (elec["currency"] == "EUR")]
elec["year"] = elec["time"].astype(str).str.slice(0, 4).astype(int)
elec = elec[elec["year"].isin(YEARS)]
elec = (elec.groupby(["geo", "year"], as_index=False)["value"].mean()
            .rename(columns={"geo": "country_code",
                             "value": "electricity_price"}))

env = read_raw("gov_10a_exp")
env = env[(env["unit"] == "PC_GDP") & (env["sector"] == "S13") &
          (env["cofog99"] == "GF05") & (env["na_item"] == "TE")]
env["time"] = env["time"].astype(int)
env = env[["geo", "time", "value"]].rename(
    columns={"geo": "country_code", "time": "year",
             "value": "environmental_expenditure"})

PRICE_COLUMN = re.compile(r"^([A-Z]{2})_price_with_tax_euro95$")

BULLETIN_TO_EUROSTAT = {"GR": "EL"}

def find_price_columns(sheet_data):
    """Номер строки с заголовками и колонки с ценой бензина по странам."""
    for row in range(min(5, len(sheet_data))):
        found = {}
        for col, value in sheet_data.iloc[row].items():
            match = PRICE_COLUMN.match(str(value).strip())
            if match:
                code = match.group(1)
                found[col] = BULLETIN_TO_EUROSTAT.get(code, code)
        if found:
            return row, found
    return None, {}

def load_fuel(path):
    xl = pd.ExcelFile(path)
    for sheet in xl.sheet_names:
        sheet_data = xl.parse(sheet, header=None)
        header_row, price_cols = find_price_columns(sheet_data)
        if not price_cols:
            continue

        price_cols = {col: code for col, code in price_cols.items()
                      if code in NAMES}

        body = sheet_data.iloc[header_row + 3:]
        dates = pd.to_datetime(body.iloc[:, 0], errors="coerce")

        long = []
        for col, code in price_cols.items():
            long.append(pd.DataFrame({
                "country_code": code,
                "date": dates,
                "price": pd.to_numeric(body[col], errors="coerce"),
            }))

        data = pd.concat(long, ignore_index=True).dropna()
        data["year"] = data["date"].dt.year
        data = data[data["year"].isin(YEARS)]

        out = data.groupby(["country_code", "year"], as_index=False)["price"].mean()
        out["fuel_price"] = out["price"] / 1000
        return out[["country_code", "year", "fuel_price"]]

fuel = load_fuel(f"{RAW}/weekly_oil_bulletin_history.xlsx")

panel = pd.MultiIndex.from_product(
    [sorted(NAMES), YEARS], names=["country_code", "year"]
).to_frame(index=False)

parts = [ev, nama, inc, pop, elec, env, fuel]

for part in parts:
    panel = panel.merge(part, on=["country_code", "year"], how="left")

panel.insert(1, "country", panel["country_code"].map(NAMES))
panel.insert(2, "region", panel["country_code"].map(REGIONS))

panel.to_csv(f"{RAW}/panel_raw.csv", index=False)

df = panel.copy()

df = df.drop_duplicates(["country_code", "year"])

df["year"] = df["year"].astype(int)
num_cols = [c for c in df.columns
            if c not in ("country_code", "country", "region", "year")]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

positive = ["new_cars_total", "gdp_per_capita_pps", "median_disposable_income",
            "aic_per_capita_pps", "population", "fuel_price",
            "electricity_price"]
for c in positive:
    df.loc[df[c].notna() & (df[c] <= 0), c] = np.nan

for c in ["new_cars_bev", "new_cars_phev", "environmental_expenditure"]:
    df.loc[df[c].notna() & (df[c] < 0), c] = np.nan

bad = df["new_cars_bev"].notna() & df["new_cars_total"].notna() \
      & (df["new_cars_bev"] > df["new_cars_total"])
df.loc[bad, "new_cars_bev"] = np.nan

interpolate = ["gdp_per_capita_pps", "aic_per_capita_pps",
               "median_disposable_income", "population",
               "electricity_price", "fuel_price"]
df = df.sort_values(["country_code", "year"])
df[interpolate] = (df.groupby("country_code")[interpolate]
                     .transform(lambda s: s.interpolate(limit=1, limit_area="inside")))

df["ev_share"] = 100 * df["new_cars_bev"] / df["new_cars_total"]
df["ev_phev_share"] = (100 * (df["new_cars_bev"] + df["new_cars_phev"])
                       / df["new_cars_total"])

df["bev_per_100k"] = 100_000 * df["new_cars_bev"] / df["population"]
df["ev_share_change_pp"] = df.groupby("country_code")["ev_share"].diff()
df["gdp_change_pct"] = (100 * df.groupby("country_code")["gdp_per_capita_pps"]
                                .pct_change())

df["ev_level"] = pd.cut(df["ev_share"],
                        bins=[-0.01, 1, 5, 15, 100],
                        labels=["до 1%", "1-5%", "5-15%", "выше 15%"])

order = ["country_code", "country", "region", "year",
         "new_cars_total", "new_cars_bev", "new_cars_phev",
         "ev_share", "ev_phev_share", "ev_level",
         "gdp_per_capita_pps", "median_disposable_income",
         "aic_per_capita_pps", "population",
         "fuel_price", "electricity_price",
         "environmental_expenditure",
         "bev_per_100k", "gdp_change_pct"]
order += [c for c in df.columns if c not in order]

clean = df[order].sort_values(["country_code", "year"]).reset_index(drop=True)
clean.to_csv(f"{CLEAN}/ev_welfare_eu.csv", index=False)

quality = pd.DataFrame({
    "n_missing": clean.isna().sum(),
    "pct_missing": (100 * clean.isna().mean()).round(1),
})
quality.to_csv(f"{CLEAN}/data_quality.csv")

print("строк:", len(clean), "| стран:", clean["country_code"].nunique())
print("\nпропуски:")
print(quality[quality["n_missing"] > 0].to_string())
print("\nпокрытие ev_share по годам:")
print(clean.groupby("year")["ev_share"].count().to_string())

строк: 270 | стран: 27

пропуски:
                    n_missing  pct_missing
new_cars_phev              93         34.4
ev_phev_share              93         34.4
gdp_change_pct             27         10.0
ev_share_change_pp         27         10.0

покрытие ev_share по годам:
year
2015    27
2016    27
2017    27
2018    27
2019    27
2020    27
2021    27
2022    27
2023    27
2024    27


## 3. Анализ и визуализация

Описательная статистика, сравнение стран и регионов, корреляционный и регрессионный анализ, визуализация результатов.

In [ ]:
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

FIG = "visualizations"
RES = "results"
os.makedirs(FIG, exist_ok=True)
os.makedirs(RES, exist_ok=True)

plt.rcParams["figure.dpi"] = 150
plt.rcParams["font.size"] = 10

df = pd.read_csv("data/cleaned/ev_welfare_eu.csv")

LAST = int(df["year"].max())
last = df[df["year"] == LAST]

notes = []

def say(title, obj=""):
    """Печатает блок и складывает его же в конспект для README."""
    text = f"\n### {title}\n{obj}" if str(obj) else f"\n### {title}"
    print(text)
    notes.append(text)

main_cols = ["ev_share", "bev_per_100k", "gdp_per_capita_pps",
             "median_disposable_income", "aic_per_capita_pps",
             "fuel_price", "electricity_price", "environmental_expenditure"]

desc = df[main_cols].describe().T[["count", "mean", "50%", "std", "min", "max"]]
desc.columns = ["наблюдений", "среднее", "медиана", "ст.откл", "минимум", "максимум"]
desc = desc.round(2)
desc.to_csv(f"{RES}/descriptive_stats.csv")
say("Описательная статистика по всей панели", desc.to_string())

freq = df["ev_level"].value_counts().sort_index()
freq_tbl = pd.DataFrame({
    "наблюдений": freq,
    "доля, %": (100 * freq / freq.sum()).round(1),
})
freq_tbl.to_csv(f"{RES}/frequency_ev_level.csv")
say("Частотная таблица по уровню электрификации (все годы)",
    freq_tbl.to_string())
say("Мода (самая частая категория)", str(df["ev_level"].mode().iloc[0]))

say(f"Разброс ev_share в {LAST} году",
    f"минимум {last['ev_share'].min():.1f}%, "
    f"максимум {last['ev_share'].max():.1f}%, "
    f"размах {last['ev_share'].max() - last['ev_share'].min():.1f} п.п., "
    f"медиана {last['ev_share'].median():.1f}%")

rank = last.dropna(subset=["ev_share"]).sort_values("ev_share", ascending=False)
top5 = rank.head(5)[["country", "region", "ev_share", "gdp_per_capita_pps"]]
bottom5 = rank.tail(5)[["country", "region", "ev_share", "gdp_per_capita_pps"]]
pd.concat([top5, bottom5]).to_csv(f"{RES}/top_bottom.csv", index=False)

say(f"Топ-5 по доле электромобилей, {LAST}", top5.round(1).to_string(index=False))
say(f"Анти-топ-5 по доле электромобилей, {LAST}",
    bottom5.round(1).to_string(index=False))

first_year = int(df["year"].min())
growth = (df[df["year"].isin([first_year, LAST])]
          .pivot(index="country", columns="year", values="ev_share")
          .dropna())
growth["изменение, п.п."] = (growth[LAST] - growth[first_year]).round(1)
growth = growth.sort_values("изменение, п.п.", ascending=False)
growth.to_csv(f"{RES}/growth_2015_2024.csv")

say(f"Рост доли электромобилей с {first_year} по {LAST}, лидеры",
    growth.head(5).round(1).to_string())

by_year = df.groupby("year")["ev_share"].agg(["mean", "median", "min", "max"]).round(2)
by_year.to_csv(f"{RES}/ev_share_by_year.csv")
say("Доля электромобилей по годам, среднее по ЕС", by_year.to_string())

pivot = (df.pivot_table(index="region", columns="year", values="ev_share",
                        aggfunc="mean").round(1))
pivot.to_csv(f"{RES}/pivot_region_year.csv")
say("Сводная таблица: средняя доля электромобилей по регионам",
    pivot.to_string())

region_share = (last.groupby("region")[["new_cars_bev", "new_cars_total"]]
                    .sum())
region_share["доля в новых BEV, %"] = (
    100 * region_share["new_cars_bev"] / region_share["new_cars_bev"].sum()).round(1)
region_share["доля в новых авто, %"] = (
    100 * region_share["new_cars_total"] / region_share["new_cars_total"].sum()).round(1)
region_share.to_csv(f"{RES}/region_shares.csv")
say(f"Доля регионов в новых регистрациях, {LAST}",
    region_share[["доля в новых BEV, %", "доля в новых авто, %"]].to_string())

factors = ["gdp_per_capita_pps", "median_disposable_income",
           "aic_per_capita_pps", "fuel_price", "electricity_price",
           "environmental_expenditure"]

rows = []
for c in factors:
    sub = df[["ev_share", c]].dropna()
    pr, pp = stats.pearsonr(sub["ev_share"], sub[c])
    sr, sp = stats.spearmanr(sub["ev_share"], sub[c])
    rows.append({"показатель": c, "n": len(sub),
                 "Pearson r": round(pr, 3), "p (Pearson)": round(pp, 4),
                 "Spearman rho": round(sr, 3), "p (Spearman)": round(sp, 4)})
corr = pd.DataFrame(rows).sort_values("Spearman rho", ascending=False)
corr.to_csv(f"{RES}/correlations.csv", index=False)
say("Связь доли электромобилей с показателями (вся панель)",
    corr.to_string(index=False))

rows = []
for y, g in df.groupby("year"):
    sub = g[["ev_share", "gdp_per_capita_pps"]].dropna()
    if len(sub) >= 10:
        rho, p = stats.spearmanr(sub["ev_share"], sub["gdp_per_capita_pps"])
        rows.append({"год": y, "Spearman rho": round(rho, 3),
                     "p": round(p, 4), "стран": len(sub)})
by_year_corr = pd.DataFrame(rows)
by_year_corr.to_csv(f"{RES}/correlation_by_year.csv", index=False)
say("Связь с ВВП на душу отдельно по годам",
    by_year_corr.to_string(index=False))

regressors = ["gdp_per_capita_pps", "electricity_price",
              "fuel_price", "environmental_expenditure"]
model_df = df[["country_code", "year", "ev_share"] + regressors].dropna()

X = model_df[regressors].copy()
X["gdp_per_capita_pps"] /= 1000
X = pd.concat([X, pd.get_dummies(model_df["year"], prefix="год",
                                 drop_first=True, dtype=float)], axis=1)
X = sm.add_constant(X)

ols = sm.OLS(model_df["ev_share"], X).fit(
    cov_type="cluster", cov_kwds={"groups": model_df["country_code"]})

with open(f"{RES}/regression.txt", "w", encoding="utf-8") as f:
    f.write(ols.summary().as_text())

coefs = pd.DataFrame({
    "коэффициент": ols.params.round(3),
    "p-value": ols.pvalues.round(4),
}).loc[["const"] + regressors]
say("Регрессия: доля электромобилей на показатели и год",
    coefs.to_string() + f"\n\nR^2 = {ols.rsquared:.3f}, "
    f"наблюдений: {int(ols.nobs)}")

fig, ax = plt.subplots(figsize=(9, 5))
for _, g in df.groupby("country_code"):
    ax.plot(g["year"], g["ev_share"], color="0.85", lw=1)
for code in rank.head(5)["country_code"]:
    g = df[df["country_code"] == code]
    ax.plot(g["year"], g["ev_share"], lw=2, label=g["country"].iloc[0])
ax.plot(by_year.index, by_year["mean"], color="black", lw=2.5, ls="--",
        label="Среднее по ЕС")
ax.set_xlabel("Год")
ax.set_ylabel("Доля BEV в новых регистрациях, %")
ax.set_title("Динамика доли электромобилей в странах ЕС, 2015–2024")
ax.legend(fontsize=8, loc="upper left")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(f"{FIG}/01_dynamics.png")
plt.close(fig)

colors = {"Северная Европа": "#2b8cbe", "Западная Европа": "#41ab5d",
          "Южная Европа": "#fdae61", "Восточная Европа": "#d7301f"}
bar = rank.sort_values("ev_share")
fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(bar["country"], bar["ev_share"],
        color=[colors.get(r, "gray") for r in bar["region"]])
ax.set_xlabel("Доля BEV в новых регистрациях, %")
ax.set_title(f"Доля электромобилей по странам ЕС, {LAST}")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()]
ax.legend(handles, colors.keys(), fontsize=8, loc="lower right")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(f"{FIG}/02_countries.png")
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5))
for region in pivot.index:
    ax.plot(pivot.columns, pivot.loc[region], marker="o", lw=2,
            color=colors.get(region, "gray"), label=region)
ax.set_xlabel("Год")
ax.set_ylabel("Средняя доля BEV, %")
ax.set_title("Средняя доля электромобилей по регионам ЕС")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(f"{FIG}/03_regions.png")
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 6))
sub = last.dropna(subset=["ev_share", "gdp_per_capita_pps"])
ax.scatter(sub["gdp_per_capita_pps"], sub["ev_share"], s=55,
           c=[colors.get(r, "gray") for r in sub["region"]])
for _, r in sub.iterrows():
    ax.annotate(r["country_code"], (r["gdp_per_capita_pps"], r["ev_share"]),
                fontsize=8, xytext=(4, 3), textcoords="offset points")
b, a = np.polyfit(sub["gdp_per_capita_pps"], sub["ev_share"], 1)
xs = np.linspace(sub["gdp_per_capita_pps"].min(), sub["gdp_per_capita_pps"].max(), 50)
ax.plot(xs, a + b * xs, color="black", ls="--", lw=1.2, label="Линия тренда")
ax.set_xlabel("ВВП на душу населения, PPS")
ax.set_ylabel("Доля BEV в новых регистрациях, %")
ax.set_title(f"Благосостояние и электрификация автопарка, {LAST}")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(f"{FIG}/04_scatter_gdp.png")
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(last["ev_share"].dropna(), bins=10, color="#2b8cbe", edgecolor="white")
ax.axvline(last["ev_share"].median(), color="#d7301f", ls="--", lw=2,
           label=f"Медиана {last['ev_share'].median():.1f}%")
ax.set_xlabel("Доля BEV в новых регистрациях, %")
ax.set_ylabel("Число стран")
ax.set_title(f"Распределение стран ЕС по доле электромобилей, {LAST}")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(f"{FIG}/05_distribution.png")
plt.close(fig)

cm = df[["ev_share"] + factors].corr(method="spearman")
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cm)), cm.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(cm)), cm.columns, fontsize=8)
for i in range(len(cm)):
    for j in range(len(cm)):
        ax.text(j, i, f"{cm.iloc[i, j]:.2f}", ha="center", va="center",
                fontsize=8,
                color="white" if abs(cm.iloc[i, j]) > 0.6 else "black")
fig.colorbar(im, ax=ax, shrink=0.8, label="Коэффициент Спирмена")
ax.set_title("Матрица корреляций показателей")
fig.tight_layout()
fig.savefig(f"{FIG}/06_correlation_matrix.png")
plt.close(fig)

with open(f"{RES}/summary.md", "w", encoding="utf-8") as f:
    f.write("# Результаты расчётов\n" + "\n".join(notes) + "\n")


### Описательная статистика по всей панели
                           наблюдений   среднее   медиана   ст.откл   минимум  максимум
ev_share                        270.0      5.37      1.82      8.39      0.02     51.34
bev_per_100k                    270.0    134.37     32.05    262.30      0.14   1901.20
gdp_per_capita_pps              270.0  33442.85  30363.70  14807.42  13586.90  97690.20
median_disposable_income        270.0  16724.63  16734.00   6045.08   4357.00  37781.00
aic_per_capita_pps              270.0  20620.65  20374.65   5879.63   9522.70  43699.30
fuel_price                      270.0      1.42      1.38      0.25      0.93      2.11
electricity_price               270.0      0.20      0.19      0.07      0.09      0.52
environmental_expenditure       270.0      0.75      0.70      0.31      0.20      1.70

### Частотная таблица по уровню электрификации (все годы)
          наблюдений  доля, %
ev_level                     
1-5%              77     28.5
5-15%          